# snapjudge training on Colab

1. Runtime → Change runtime type → **T4 GPU** (free tier)
2. Run top to bottom
3. Output lands in `out/`; figures are PNG+SVG in `out/figures`

**Note:** the trainer is C++/BLAS (CPU), not GPU-computed. The T4 selection just gets a solid Colab box; the GPU itself isn't used by the training loop.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
print('cpu threads:', os.cpu_count())


## Dependencies


In [ ]:
!apt-get update -qq && apt-get install -y -qq libpcre2-dev libcurl4-openssl-dev libopenblas-dev cmake git >/dev/null
!cmake --version | head -1


## Clone snapjudge


In [ ]:
!git clone --depth 1 https://github.com/Cyrax321/snapjudge.git
%cd snapjudge
!ls


## Build (CPU path, OpenBLAS on Colab)


In [ ]:
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release > /dev/null
!cmake --build build -j $(nproc)
!ls build/snapjudge*


## Sanity: all suites green


In [ ]:
!ctest --test-dir build --output-on-failure


## Train (full epoch, 1,200 train states)


In [ ]:
# Pre-download the real base checkpoint (ModernBERT-large backbone, ~800MB).
# snapshot_download gives a plain dir the trainer loads directly.
from huggingface_hub import snapshot_download
snapshot_download(repo_id='convaiinnovations/laya-typed-decisions',
                  local_dir='/content/base',
                  allow_patterns=['rl_agent_config.json','model.safetensors',
                                  'tokenizer/*','encoder/*'])
print('base ready')


In [ ]:
!./build/snapjudge-train --base /content/base --data data/train.jsonl --val data/val.jsonl --out out/typed-ft --epochs 1


## Evaluate


In [ ]:
!./build/snapjudge-eval --ckpt out/typed-ft --data data/val.jsonl --json out/report.json
!python3 -c "import json; r=json.load(open('out/report.json')); print({k:v for k,v in r.items() if k!='points'})"


## Figures


In [ ]:
!./build/snapjudge-plots --report out/report.json --out out/figures
!ls out/figures


## Look at the figures


In [ ]:
from IPython.display import Image, display
for f in ['calibration','roc','workflows','confusion-choice','confusion-score','confusion-noul','score-spread','pr-noul']:
    p = f'out/figures/{f}.png'
    try:
        display(Image(filename=p))
        print(p)
    except FileNotFoundError:
        pass


## Download / publish


In [ ]:
!cd out && zip -qr /content/snapjudge-out.zip typed-ft figures report.json
from google.colab import files
files.download('/content/snapjudge-out.zip')


## (Optional) Push the checkpoint to the HF hub


In [ ]:
import os
# os.environ['HF_TOKEN'] = 'hf_...'
import subprocess
r = subprocess.run(['./build/snapjudge-train-push','--ckpt','out/typed-ft','--repo','you/snapjudge-typed-ft'], capture_output=True, text=True, env=os.environ)
print(r.stdout)
print(r.stderr)
print('rc', r.returncode)
